# Dataset Statistics

This notebook analyzes the genre distribution of the available WAV files.

The statistics are used to describe the available dataset sizes and genre distributions for the FMA subsets.

In [ ]:
import sys
from pathlib import Path

current_path = Path.cwd()

for parent in [current_path] + list(current_path.parents):
    if (parent / "src").exists():
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find project root containing 'src' folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import os
import pandas as pd

from src.config import (
    BASE_DIR,
    PROCESSED_TRACKS_CSV
)

In [ ]:
def get_genre_distribution(audio_dir, tracks_csv):
    """
    Count available audio files per top-level genre.

    Parameters
    ----------
    audio_dir : str or Path
        Directory containing WAV files.
    tracks_csv : str or Path
        Path to prepared tracks metadata.

    Returns
    -------
    pandas.Series
        Genre distribution including missing values.
    """
    tracks = pd.read_csv(tracks_csv)

    tracks = tracks[
        [
            "track_id",
            "track_genre_top"
        ]
    ].copy()

    tracks["track_id"] = tracks["track_id"].apply(lambda x: f"{int(x):06d}")
    tracks["track_id"] = tracks["track_id"].astype(str)

    files = [
        file for file in os.listdir(audio_dir)
        if file.lower().endswith(".wav")
    ]

    df_files = pd.DataFrame(
        {
            "track_id": [os.path.splitext(file)[0] for file in files],
            "filename": files,
            "filepath": [os.path.join(audio_dir, file) for file in files]
        }
    )

    df = df_files.merge(
        tracks,
        on="track_id",
        how="left"
    )

    return df["track_genre_top"].value_counts(dropna=False)

In [ ]:
audio_dirs = {
    "small": BASE_DIR / "wav_small",
    "medium": BASE_DIR / "wav_medium",
    "large": BASE_DIR / "wav_large"
}

In [ ]:
genre_distributions = {}

for subset_name, audio_dir in audio_dirs.items():
    print(f"\n{subset_name.upper()} dataset")
    print("-" * 40)

    distribution = get_genre_distribution(
        audio_dir=audio_dir,
        tracks_csv=PROCESSED_TRACKS_CSV
    )

    genre_distributions[subset_name] = distribution

    print(distribution)

In [ ]:
genre_distribution_df = pd.DataFrame(genre_distributions).fillna(0).astype(int)

genre_distribution_df

In [ ]:
output_path = PROJECT_ROOT / "results" / "metrics" / "dataset_genre_distribution.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)

genre_distribution_df.to_csv(output_path)

print(f"Saved to: {output_path}")